# Lecture 4 DM1590: Supervised Learning, part 1

Let's first set up the environment and import the necessary libraries

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np

# for plotting in Ipython, usually the pyplot module is loaded:
import matplotlib.pyplot as plt
import sklearn
from scipy import sparse
import pandas as pd
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "mglearn"])
#import mglearn 
%matplotlib inline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/bobs/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/bobs/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/bobs/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/Users/bobs/ana

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [7]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets 



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/bobs/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/bobs/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/bobs/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/Users/bobs/ana

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

## Let's import the Iris dataset and look at its statistics

In [ ]:
from sklearn.datasets import load_iris
iris_dataset = load_iris()

X = iris_dataset['data'] # observations
Y = iris_dataset['target'] # classes

# create dataframe from data in X
# label the columns using the strings in iris_dataset.feature_names
iris_dataframe = pd.DataFrame(X, columns=iris_dataset.feature_names)
iris_dataframe.describe()

## Let's visualize the observations with a scatter plot matrix

In [ ]:
pd.plotting.scatter_matrix(iris_dataframe, c=Y, figsize=(8, 8),
                           marker='o', hist_kwds={'bins': 20}, s=60,
                           alpha=.5, cmap=plt.cm.brg, label=list(iris_dataset.target_names))

handles = [plt.plot([],[],color=plt.cm.brg(i/2.), ls="", marker=".", 
                    markersize=10)[0] for i in range(3)]
plt.legend(handles, list(iris_dataset.target_names), loc=(1.02,0))
plt.show()

## Our first machine learning model: the decision tree

In [ ]:
from sklearn import tree
clf = tree.DecisionTreeClassifier() # instantiate
clf = clf.fit(X, Y) # train

## What does this tree look like?

In [ ]:
import graphviz # make sure you have graphviz installed!!!
dot_data = tree.export_graphviz(clf, out_file=None) 
graph = graphviz.Source(dot_data) 


dot_data = tree.export_graphviz(clf, out_file=None, 
                     feature_names=iris_dataset.feature_names,  
                     class_names=iris_dataset.target_names,  
                     filled=True, rounded=True,  
                     special_characters=True)  
graph = graphviz.Source(dot_data)  
graph.render("iris") 

graph

## How do we know how well it worked? 

Oops. We have used all our data to create the tree. 
- Either, we need to go collect more in order to test it.
- Or we simulate doing this with the data we have!

## Partition the dataset

Let us divide our data into two sets: one for training (80%) and one for testing (20%)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, train_size=0.8)
print(y_test)

## Now train tree on training dataset

In [ ]:
clf = tree.DecisionTreeClassifier()
clf = clf.fit(X_train, y_train)

## Now test tree with the testing set

In [ ]:
print("Test set preds :", clf.predict(X_test))
print("Test set labels:", y_test)

## How well did it do?

Let's summarize its performance with a variety of metrics.

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, clf.predict(X_test),target_names=iris_dataset.target_names))

Let's also look at the confusion table.

In [ ]:
def print_cm(cm, labels, hide_zeroes=False, hide_diagonal=False, hide_threshold=None):
    """pretty print for confusion matrixes"""
    """taken and adapted by Bob Sturm from https://gist.github.com/zachguo/10296432"""
    columnwidth = max([len(x) for x in labels] + [5])  # 5 is value length
    empty_cell = " " * columnwidth
    
    # Begin CHANGES
    fst_empty_cell = (columnwidth-3)//2 * " " + "t\p" + (columnwidth-3)//2 * " "
    
    if len(fst_empty_cell) < len(empty_cell):
        fst_empty_cell = " " * (len(empty_cell) - len(fst_empty_cell)) + fst_empty_cell
    # Print header
    print("    " + fst_empty_cell, end=" ")
    # End CHANGES
    
    for label in labels:
        print("%{0}s".format(columnwidth) % label, end=" ")
        
    print()
    # Print rows
    for i, label1 in enumerate(labels):
        print("    %{0}s".format(columnwidth) % label1, end=" ")
        for j in range(len(labels)):
            cell = "%{0}.2f".format(columnwidth) % cm[i, j]
            if hide_zeroes:
                cell = cell if float(cm[i, j]) != 0 else empty_cell
            if hide_diagonal:
                cell = cell if i != j else empty_cell
            if hide_threshold:
                cell = cell if cm[i, j] > hide_threshold else empty_cell
            print(cell, end=" ")
        print()

from sklearn.metrics import confusion_matrix
print_cm(confusion_matrix(y_test, clf.predict(X_test)),iris_dataset.target_names)